<a href="https://colab.research.google.com/github/juanjodoblasm/proyecto_algoritmos_optimizacion/blob/main/SEMINARIO/trabajo_practico-problema1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Proyecto de programación
Nombre y Apellidos: Christian Dario Barahona Pesantes y Juan José Doblas Martínez

[GitHub](https://github.com/juanjodoblasm/proyecto_algoritmos_optimizacion)

Problema:
1. Sesiones de doblaje

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:
- Número de actores: 10
- Número de tomas: 30
- Actores/Tomas: https://bit.ly/36D8IuK
  - 1 indica que el actor participa en la toma,
  - 0 en caso contrario.

Librerías que usaremos durante el trabajo:

In [2]:
import math
from functools import lru_cache
import pandas as pd

(*)¿Cuántas posibilidades hay sin tener en cuenta las restricciones?

Respuesta

Para saber cuántas posibilidades existen sin tener en cuenta las restricciones, debemos contar el número de particiones de un conjunto de 30 elementos (el conjunto de tomas), ya que queremos saber en cuántas formas podemos dividir las 30 tomas en hasta 30 días. Esto es porque suponemos que cada día se rueda al menos una toma hasta finalizar el rodaje. Además, también consideramos que si seleccionamos dos o más tomas en un día, no nos importa el orden en el que se rueden y tampoco nos importa el día que se asigne a cada grupo de tomas. Al final, lo relevante es la función de coste, que es el número de actores que llevamos al set de rodaje cada día y esto no cambia con el orden de las tomas de un mismo día o con el día que asignemos a cada grupo de tomas.

Este número se conoce como el trigésimo número de Bell, $B_{30}$. Los números de Bell siguen la siguiente fórmula recursiva:
$$
\begin{align*}
    B_0 &= 1, \\
    B_n &= \sum_{k=0}^{n-1}\binom{n-1}{k}B_k, \forall n \ge 1.
\end{align*}
$$
Esta fórmula puede explicarse observando que, a partir de una partición arbitraria de $n$ elementos, la eliminación del conjunto que contiene un elemento fijado deja una partición de un conjunto menor de $k$ elementos para algún número $k \in \{0, 1, \ldots, n-1\}$. Hay $\binom{n-1}{k}$ opciones para los $k$ elementos que quedan después de que se elimine un conjunto, y $B_k$ opciones de cómo dividirlos.

Así, podemos calcular el número de posibilidades con el siguiente código.

In [3]:
# Recurrencia de los números de Bell
@lru_cache(maxsize=31)
def bell_number(n):
  if n == 0:
    return 1
  else:
    total = 0
    for k in range(n):
      total += math.comb(n-1, k) * bell_number(k)
    return total

bell_number(30)

846749014511809332450147

Esto es, aproximadamente 847 mil trillones de posibilidades, es decir, $847\cdot10^{21}$.

¿Cuántas posibilidades hay teniendo en cuenta todas las restricciones?

Respuesta

La única restricción que tenemos nos dice que ningún subconjunto de la partición del conjunto de tomas puede superar los 6 elementos.

De forma general, podemos definir una modificación del número de Bell que cuente el número de particiones posibles de un conjunto de $n$ elementos con, como máximo, $m$ elementos por subconjunto. Esto también se puede definir de manera recursiva:
$$
\begin{align*}
    B_0 &= 1, \\
    B_n^{(m)} &= \sum_{k=\max(n-m, 0)}^{n-1}\binom{n-1}{k}B_k^{(m)}, \forall n \ge 1.
\end{align*}
$$
En este caso, si $n \le m$, la fórmula es la misma que la del número de Bell, ya que la restricción se cumple trivialmente. Por otro lado, cuando $n > m$, observamos que el sumatorio solamente recorre $m$ términos. Esto se debe a que, a partir de una partición arbitraria de $n$ elementos, la eliminación del conjunto que contiene un elemento fijado deja una partición restringida de un conjunto menor de $k$ elementos para algún número $k$, a priori entre $0$ y $n-1$. Sin embargo, el conjunto eliminado, que tiene $n-k$ elementos, tampoco puede tener más de $m$ elementos, por lo que se cumple la desigualdad $n-k \le m$, de donde $k \ge n-m$; junto con lo anterior, obtenemos $n-m \le k \le n-1$.

De esta forma, hay $\binom{n-1}{k}$ opciones para los $k$ elementos que quedan después de que se elimine un conjunto, y $B_k^{(m)}$ opciones de cómo dividirlos.

Así, podemos calcular el número de posibilidades con el siguiente código.

In [4]:
# Recurrencia de los números de Bell modificados
@lru_cache(maxsize=31)
def bell_number_modified(n, m):
  if n == 0:
    return 1
  else:
    total = 0
    for k in range(max(n-m, 0), n):
      total += math.comb(n-1, k) * bell_number_modified(k, m)
    return total

bell_number_modified(30, 6)

726391948970868949621309

Esto es, aproximadamente 726 mil trillones de posibilidades, es decir, $726\cdot10^{21}$.

Modelo para el espacio de soluciones

(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Arguméntalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguméntalo)


Respuesta

El objetivo del problema es planificar las sesiones por día, es decir, decidir una partición del conjunto de 30 tomas con la restricción de que no puede haber más de 6 tomas por día (subconjunto). Esto significa que no nos importa el orden de las tomas que se rueden en un mismo día, ni tampoco el día en que se ruede cada grupo de tomas, ya que la función de coste no cambia con esas variaciones. Por eso, la estructura de datos que mejor se adapta a organizar las tomas es el `set`.

Por otro lado, para tratar con los actores, también usaremos `sets`, porque dada una agrupación de tomas, para saber qué actores debemos llevar al estudio de grabación ese día, hay que hacer la unión de los conjuntos de actores que participan en cada una de esas tomas.

Según el modelo para el espacio de soluciones:

(*)¿Cual es la función objetivo?

Respuesta

La función objetivo nos debe dar el gasto por los servicios de los actores de doblaje. Como todos los actores cobran la misma cantidad, podemos suponer que cada actor tiene un coste $1$.

Como el número de actores que debemos llevar al estudio de grabación depende únicamente de las tomas que hayamos decidido grabar ese día y a cada actor le asignamos coste $1$, nuestra función objetivo tomará como variable la partición de las tomas y nos devolverá el coste asociado a esa partición sumando el número de actores que se requieren en cada conjunto de tomas, que se calcula haciendo la unión de los actores que participan en cada toma de ese conjunto.

Formalmente, sean $T = \{t_i | i = 1, \ldots, n\}$ el conjunto de tomas y $A = \{a_j | j = 1, \ldots, p\}$ el conjunto de actores. Definimos $A_i \subseteq A$ como el subconjunto de actores que participan en la toma $t_i$. También definimos $\mathbb{P}^{(m)}(T)$ como el conjunto de todas las particiones de $T$ con, como máximo, $m$ elementos por subconjunto (por lo que $\left|\mathbb{P}^{(m)}(T)\right| = B_n^{(m)}$). Nuestra función objetivo es
$$
f : \mathbb{P}^{(6)}(T) \to \mathbb{N},
$$
definida como
$$
f\left(\{G_1, \ldots, G_q\}\right) = \sum_{k = 1}^q\left|\bigcup_{t_i \in G_k}A_i\right|
$$

(*)¿Es un problema de maximización o minimización?

Respuesta

El objetivo del problema nos pide que el gasto por los servicios de los actores de doblaje sea el menor posible, por lo tanto, es un problema de minimización.

Formalmente, dado que lo que buscamos no es únicamente el valor mínimo del coste sino la partición concreta que lo alcanza, buscamos
$$
\underset{P \in \mathbb{P}^{(6)}(T)}{\arg\min}\ f(P)
$$

Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

In [6]:
def generar_particiones(elementos_restantes):
    elementos_restantes = set(elementos_restantes)
    if not elementos_restantes:
        yield []
        return

    fijada = next(iter(elementos_restantes))
    resto = elementos_restantes - {fijada}

    for i in range(min(5, len(resto)) + 1):
        for companeras in combinations(resto, i):
            grupo = {fijada} | set(companeras)
            nuevos_restantes = resto - set(companeras)


In [9]:
# 1. Cargar datos del CSV (fila 1 como cabecera)
url = 'https://raw.githubusercontent.com/juanjodoblasm/proyecto_algoritmos_optimizacion/main/SEMINARIO/data_doblaje.csv'
df = pd.read_csv(url, header=1)

# 2. Limpieza de datos
# Quitamos la fila en blanco y la fila TOTAL y transformamos la columna a int
df['Toma'] = pd.to_numeric(df['Toma'], errors='coerce')
df = df.dropna(subset=['Toma'])
df['Toma'] = df['Toma'].astype(int)
# Guardamos la columna Toma como índice
df = df.set_index('Toma')
# Nos quedamos solo con las columnas de actores
df = df[[str(i) for i in range(1, 11)]]
# Columnas como enteros 1..10, con nombre "Actor"
df.columns = df.columns.astype(int)
df.columns.name = 'Actor'
# Los datos del dataframe los guardamos como enteros
df = df.astype(int)

# 3. Creación de conjuntos para cada toma con los actores que deben intervenir en cada una
tomas = {toma: set(row.index[row == 1]) for toma, row in df.iterrows()}

In [13]:
def fuerza_bruta(tomas):
  # Preparamos el conjunto de tomas e inicializamos el plan de tomas diarias
  tomas_pendientes = set(tomas.keys())
  planificacion = []

  while tomas_pendientes:
    hoy = []
    actores_hoy = set()

    # Se debe llenar el día actual con un máx de 6 tomas
    while len(hoy) < 6 and tomas_pendientes:
      mejor_toma = None
      menor_incremento = float('inf')

      for t in tomas_pendientes:
        if not hoy:
          # Regla para el primer día conseguir la toma que más actores tenga
          incremento = -len(tomas[t])
        else:
          # Regla para el resto de días y conseguir los que añadan menos actores nuevos
          nuevos_actores = tomas[t] - actores_hoy
          incremento = len(nuevos_actores)

        if incremento < menor_incremento:
          menor_incremento = incremento
          mejor_toma = t

      # Añadimos a la lista la mejor toma detectada
      hoy.append(mejor_toma)
      # Operación equivalente a la unión de cojuntos. Se añaden solamente los actores extra
      actores_hoy.update(tomas[mejor_toma])
      # Se elimina del conjunto de tomas pendientes por grabar la toma que ha sido seleccionada
      tomas_pendientes.remove(mejor_toma)

    planificacion.append(hoy)

  return planificacion

In [20]:
def coste_dia(dia_tomas, tomas):
    # Control para días vacíos, una lista sin tomas
    if not dia_tomas: return 0
    # Lista de actores que intervendrán en un día concreto
    actores_unidos = set()
    # Recorremos las tomas programadas para el día
    for t in dia_tomas:
        # Añadimos los nuevos actores que intervienen en la toma t
        actores_unidos.update(tomas[t])
    return len(actores_unidos)

def coste_total(planificacion, tomas):
    # Recorremos cada día de la planificación calculando su coste y sumando el total
    return sum(coste_dia(dia, tomas) for dia in planificacion)

In [21]:
planificacion = fuerza_bruta(tomas)
print(planificacion)
coste_total(planificacion, tomas)

[[1, 2, 6, 7, 9, 13], [11, 17, 19, 23, 3, 4], [12, 8, 14, 18, 22, 24], [10, 15, 21, 5, 28, 30], [20, 27, 16, 25, 26, 29]]


31

Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

(*)Calcula la complejidad del algoritmo

Respuesta

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Aplica el algoritmo al juego de datos generado

Respuesta

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta